# **Proyecto Etapa 3 — Aprendizaje Supervisado y No Supervisado con PySpark**

### **Curso: TC5057 · Análisis de Grandes Volúmenes de Datos**
#### **Tecnológico de Monterrey**
##### **Profesor Titular: Dr. Iván Olmos Pineda**

---

### **Equipo #17**
#### **Tutor: José Carlos Soto**

| Nombre | Matrícula |
|--------|-----------|
| Diego Falcón Costilla | A00000000 |

---

**Dataset:** GTEx Analysis V10 — Gene Expression TPM (NIH / Broad Institute)  
**Archivo:** `GTEx_Analysis_2022-06-06_v10_RNASeQCv2.4.2_gene_tpm_non_lcm.gct`  
**Genes:** 59,033 | **Muestras RNASEQ:** 19,788 | **Donantes:** 981

---
## 1. Construcción de la muestra M

### 1.1 Definición de M

La muestra M se define como el conjunto de **todas** las muestras RNASEQ disponibles en GTEx V10, organizadas en las **10 particiones** derivadas de las variables de caracterización:

$$M = \{M_i : M_i \text{ es una partición de TISSUE\_GROUP} \times \text{SEX\_LABEL}\},\quad i = 1, \ldots, 10$$

con 5 grupos de tejido × 2 sexos biológicos como ejes de particionamiento.

### 1.2 Evaluación de representatividad y mejoras implementadas

Se evaluó si la muestra M es suficientemente representativa de la población P (donantes adultos del proyecto GTEx V10 con datos RNASEQ). El análisis identificó los siguientes ajustes necesarios respecto a versiones previas de trabajo:

| Aspecto evaluado | Versión previa | Versión actual | Justificación |
|-----------------|----------------|----------------|---------------|
| **Cobertura de muestras** | Sub-muestra reducida (1/100 de columnas) | Todas las 19,788 muestras RNASEQ | La tarea exige trabajar con la muestra lo más completa posible |
| **Selección de genes** | Sub-selección aleatoria de columnas | Top-500 genes por varianza inter-muestras | Los genes de mayor varianza son los más discriminantes biológicamente (Law et al., 2016) |
| **División train/test** | Aleatoria por muestra (sesgo intra-donante) | Por donante (`SUBJID`) — garantiza `Tri ∩ Tsi = ∅` | Evita que el modelo aprenda perfiles individuales en lugar de patrones generalizables |
| **Cobertura de particiones** | 2–4 particiones | 10 particiones completas | M debe representar toda la diversidad tisular de P |
| **SMTSD disponible** | No incluido | Incluido en metadatos | Permite clasificación fina por sub-tipo de tejido si se requiere |

**Conclusión de representatividad:** la muestra M con 19,616 muestras cubre los 946 donantes únicos disponibles, representa los 5 grupos de tejido y ambos sexos biológicos, y selecciona los genes de mayor varianza para maximizar la señal discriminante. Se considera suficientemente representativa para el análisis de aprendizaje automático de esta etapa.

### 1.3 Estrategia de carga del dataset

El archivo TPM tiene **59,033 genes × 19,788 muestras** (~4.7 GB). Se aplica una estrategia en dos pasos para mantener factibilidad computacional sin perder representatividad biológica:

1. **Cálculo de varianza por bloques:** lectura chunked (500 genes/bloque) con `pandas.read_csv`, ~80 MB pico por bloque. Varianza calculada de forma vectorizada con `DataFrame.var(axis=1)`.
2. **Carga de la matriz final:** solo las 500 filas de los top genes, para todas las 19,616 muestras → 78.5 MB en memoria.

Los metadatos (particiones, donantes, sexo) se cargan y procesan con **PySpark** para aprovechar el procesamiento distribuido en la unión y filtrado de los 19,788 registros.

In [1]:
import sys, os

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

_conda_env = os.path.dirname(sys.executable)
_java_home = os.path.join(_conda_env, 'Library', 'lib', 'jvm')
if os.path.isdir(_java_home):
    os.environ['JAVA_HOME'] = _java_home

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType

sys.path.insert(0, os.path.abspath('../src'))
from GlobalVariables import (
    FILE_PATH, SAMPLE_ATTRS_PATH, SUBJECT_PHENO_PATH,
    N_GENES, RANDOM_SEED
)

import random
import numpy as np
import pandas as pd
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'JAVA_HOME: {os.environ.get("JAVA_HOME", "ERROR")}')
print(f'Semilla aleatoria: {RANDOM_SEED}')
print(f'Genes en el dataset: {N_GENES:,}')

JAVA_HOME: C:\Users\diego\anaconda3\envs\big-data\Library\lib\jvm
Semilla aleatoria: 42
Genes en el dataset: 59,033


In [2]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('GTEx_Etapa3_TC5057') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print(f'Workers: {spark.sparkContext.defaultParallelism}')

Spark version: 4.1.1
Workers: 32


In [3]:
# --- Carga de metadatos: todos los atributos de las muestras RNASEQ ---
sa_df = spark.read.csv(SAMPLE_ATTRS_PATH, sep='\t', header=True) \
    .select('SAMPID', 'SMTS', 'SMTSD', 'SMAFRZE') \
    .filter(F.col('SMAFRZE') == 'RNASEQ') \
    .withColumn('SUBJID', F.regexp_extract(F.col('SAMPID'), r'^(GTEX-[^-]+)', 1))

sp_df = spark.read.csv(SUBJECT_PHENO_PATH, sep='\t', header=True) \
    .select('SUBJID', 'SEX')

# Mapeo de SMTS a grupos de tejido (misma nomenclatura que Etapa 2)
tissue_group_col = (
    F.when(F.col('SMTS').isin('Brain', 'Nerve'), 'Nervioso')
     .when(F.col('SMTS').isin('Blood', 'Bone Marrow', 'Spleen'), 'Hematopoyetico')
     .when(F.col('SMTS').isin('Heart', 'Blood Vessel'), 'Cardiovascular')
     .when(F.col('SMTS').isin('Muscle', 'Adipose Tissue', 'Skin'), 'Musculoesqueletico')
     .otherwise('Visceral_Metabolico')
)
sex_label_col = F.when(F.col('SEX') == '1', 'Masculino').otherwise('Femenino')

meta_df = sa_df.join(sp_df, on='SUBJID', how='inner') \
    .withColumn('TISSUE_GROUP', tissue_group_col) \
    .withColumn('SEX_LABEL', sex_label_col) \
    .withColumn('COL_NAME',
        F.regexp_replace(F.regexp_replace(F.col('SAMPID'), '-', '_'), '\\.', '_'))

total_samples = meta_df.count()
total_donors  = meta_df.select('SUBJID').distinct().count()
print(f'Muestras RNASEQ en M: {total_samples:,}')
print(f'Donantes únicos     : {total_donors:,}')

Muestras RNASEQ en M: 19,788
Donantes únicos     : 946


In [4]:
# --- Distribución de las 10 particiones de M ---
print('Distribución de M por partición (TISSUE_GROUP × SEX_LABEL):')
partition_dist = meta_df.groupBy('TISSUE_GROUP', 'SEX_LABEL') \
    .count() \
    .orderBy('TISSUE_GROUP', 'SEX_LABEL')
partition_dist.show(20)

print('Distribución por grupo de tejido (total):')
meta_df.groupBy('TISSUE_GROUP').count().orderBy('count', ascending=False).show()

Distribución de M por partición (TISSUE_GROUP × SEX_LABEL):


+-------------------+---------+-----+
|       TISSUE_GROUP|SEX_LABEL|count|
+-------------------+---------+-----+
|     Cardiovascular| Femenino|  771|
|     Cardiovascular|Masculino| 1573|
|     Hematopoyetico| Femenino|  475|
|     Hematopoyetico|Masculino|  932|
| Musculoesqueletico| Femenino| 1349|
| Musculoesqueletico|Masculino| 2827|
|           Nervioso| Femenino| 1066|
|           Nervioso|Masculino| 2838|
|Visceral_Metabolico| Femenino| 2864|
|Visceral_Metabolico|Masculino| 5093|
+-------------------+---------+-----+

Distribución por grupo de tejido (total):


+-------------------+-----+
|       TISSUE_GROUP|count|
+-------------------+-----+
|Visceral_Metabolico| 7957|
| Musculoesqueletico| 4176|
|           Nervioso| 3904|
|     Cardiovascular| 2344|
|     Hematopoyetico| 1407|
+-------------------+-----+



### 1.2 Carga del dataset TPM y selección de genes por varianza

El archivo TPM tiene **59,033 genes (filas) × 19,788 muestras (columnas)**. Cargar la matriz completa requeriría ~4.7 GB en memoria. Para mantener factibilidad computacional sin perder representatividad biológica, se aplica una estrategia en dos pasos:

1. **Cálculo de varianza** por gen a través de todas las muestras válidas: lectura por bloques (`chunksize=500` genes) para mantener el pico de memoria en ~80 MB/bloque.
2. **Carga de la matriz final**: solo las filas correspondientes a los top-500 genes, para todas las 19,788 muestras → ~75 MB en memoria.

La selección por varianza está validada en la literatura de RNA-seq (Law et al., 2016) y en nuestros propios experimentos de Tarea 3: los top-500 genes por varianza contienen los marcadores tisulares más discriminantes (PLN, ACTN2, MYL7, ACTC1, etc.).

In [5]:
# --- Paso 1: obtener nombres de columna del archivo TPM ---
# Nota: el archivo GTEx usa guiones en los IDs de muestra (GTEX-1117F-...)
# pero Spark sanitiza a guiones bajos (GTEX_1117F_...). Se construye un mapeo bidireccional.
import pandas as pd

peek = pd.read_csv(FILE_PATH, sep='\t', skiprows=2, nrows=0)
all_file_cols = peek.columns.tolist()

# Mapeo: nombre_en_archivo (guiones) -> nombre_sanitizado (guiones bajos)
file_to_san = {
    c: c.replace('-', '_').replace('.', '_')
    for c in all_file_cols
    if c not in ('Name', 'Description')
}

meta_col_names = set(
    row['COL_NAME'] for row in meta_df.select('COL_NAME').collect()
)

# Columnas válidas: aquellas cuyo nombre sanitizado está en los metadatos
valid_sample_cols_file = [c for c, san in file_to_san.items() if san in meta_col_names]
rename_dash_to_san     = {c: file_to_san[c] for c in valid_sample_cols_file}

print(f'Columnas en el archivo TPM       : {len(all_file_cols):,}')
print(f'Columnas de muestra en el archivo: {len(file_to_san):,}')
print(f'Columnas en M (metadatos)        : {len(meta_col_names):,}')
print(f'Columnas válidas (intersección)  : {len(valid_sample_cols_file):,}')
print(f'Ejemplo mapeo: {list(rename_dash_to_san.items())[0]}')

Columnas en el archivo TPM       : 19,618
Columnas de muestra en el archivo: 19,616
Columnas en M (metadatos)        : 19,788
Columnas válidas (intersección)  : 19,616
Ejemplo mapeo: ('GTEX-1117F-0005-SM-HL9SH', 'GTEX_1117F_0005_SM_HL9SH')


In [6]:
# --- Paso 2: calcular varianza por gen (lectura por bloques) ---
# Cada bloque: 500 genes × ~19,616 muestras ≈ 80 MB pico en memoria
# Varianza calculada de forma vectorizada con DataFrame.var(axis=1) — evita iterrows()
CHUNK_SIZE = 500
gene_vars  = {}
usecols_file = ['Name'] + valid_sample_cols_file

n_chunks = -(-N_GENES // CHUNK_SIZE)  # ceil division
print(f'Calculando varianza para {N_GENES:,} genes en ~{n_chunks} bloques de {CHUNK_SIZE}...')

for i, chunk in enumerate(pd.read_csv(
        FILE_PATH, sep='\t', skiprows=2,
        chunksize=CHUNK_SIZE, usecols=usecols_file)):
    chunk = chunk.rename(columns=rename_dash_to_san).set_index('Name')
    chunk = chunk.apply(pd.to_numeric, errors='coerce').fillna(0.0)
    gene_vars.update(chunk.var(axis=1).to_dict())
    if (i + 1) % 20 == 0:
        print(f'  Bloque {i+1:>3} procesado ({(i+1)*CHUNK_SIZE:,} genes)')

gene_var_series = pd.Series(gene_vars).sort_values(ascending=False)
print(f'\nVarianza calculada para {len(gene_var_series):,} genes.')
print('Top 10 genes por varianza:')
print(gene_var_series.head(10))

Calculando varianza para 59,033 genes en ~119 bloques de 500...


  Bloque  20 procesado (10,000 genes)


  Bloque  40 procesado (20,000 genes)


  Bloque  60 procesado (30,000 genes)


  Bloque  80 procesado (40,000 genes)


  Bloque 100 procesado (50,000 genes)



Varianza calculada para 59,033 genes.
Top 10 genes por varianza:
ENSG00000244734.4     3.018102e+09
ENSG00000210082.2     6.927912e+08
ENSG00000198804.2     6.029021e+08
ENSG00000198712.1     4.503598e+08
ENSG00000198938.2     4.438837e+08
ENSG00000188536.13    4.395546e+08
ENSG00000198886.2     3.579125e+08
ENSG00000198899.2     3.578467e+08
ENSG00000275896.7     2.180929e+08
ENSG00000163220.11    2.052547e+08
dtype: float64


In [7]:
# --- Paso 3: seleccionar top-500 genes y cargar su matriz completa ---
TOP_N_GENES = 500
top_gene_ids = gene_var_series.head(TOP_N_GENES).index.tolist()

target_genes = set(top_gene_ids)
collected    = []

print(f'Cargando {TOP_N_GENES} genes seleccionados x {len(valid_sample_cols_file):,} muestras...')

for chunk in pd.read_csv(
        FILE_PATH, sep='\t', skiprows=2,
        chunksize=CHUNK_SIZE, usecols=usecols_file):
    chunk = chunk.rename(columns=rename_dash_to_san)
    hit = chunk[chunk['Name'].isin(target_genes)]
    if not hit.empty:
        collected.append(hit)

tpm_top = pd.concat(collected).set_index('Name')
tpm_top = tpm_top.apply(pd.to_numeric, errors='coerce').fillna(0.0)

print(f'Matriz de genes seleccionados: {tpm_top.shape[0]} genes × {tpm_top.shape[1]} muestras')
print(f'Tamaño en memoria: {tpm_top.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'Ejemplo columna: {tpm_top.columns[0]}')

Cargando 500 genes seleccionados x 19,616 muestras...


Matriz de genes seleccionados: 500 genes × 19616 muestras


Tamaño en memoria: 78.5 MB
Ejemplo columna: GTEX_1117F_0005_SM_HL9SH


In [8]:
# --- Construcción de la matriz de features M: muestras × genes ---
# tpm_top index = gene IDs (Ensembl, con puntos)
# tpm_top columns = sample COL_NAMEs (ya en formato guiones bajos, desde cell-09)

# Sanitizar nombres de genes: reemplazar '.' por '_'
rename_genes = {g: g.replace('.', '_') for g in tpm_top.index}
tpm_top_san  = tpm_top.rename(index=rename_genes)

# Filtrar a los top genes que se cargaron
top_gene_ids_san = [g.replace('.', '_') for g in top_gene_ids if g in tpm_top.index]

# Transponer: filas=muestras, columnas=genes
tpm_T = tpm_top_san.loc[top_gene_ids_san].T.reset_index()
tpm_T = tpm_T.rename(columns={'index': 'COL_NAME'})
gene_cols = [c for c in tpm_T.columns if c != 'COL_NAME']

# Unir con metadatos (ambos usan formato guiones bajos en COL_NAME)
meta_pd = meta_df.select(
    'COL_NAME', 'TISSUE_GROUP', 'SEX_LABEL', 'SMTSD', 'SUBJID'
).toPandas()
M = tpm_T.merge(meta_pd, on='COL_NAME', how='inner')
M = M.dropna(subset=['TISSUE_GROUP', 'SUBJID'])
M[gene_cols] = M[gene_cols].fillna(0.0)

print(f'Muestra M final: {M.shape[0]:,} muestras × {len(gene_cols)} genes')
print(f'Donantes únicos en M: {M["SUBJID"].nunique():,}')
print('\nDistribución TISSUE_GROUP:')
print(M['TISSUE_GROUP'].value_counts())

Muestra M final: 19,616 muestras × 500 genes
Donantes únicos en M: 946

Distribución TISSUE_GROUP:
TISSUE_GROUP
Visceral_Metabolico    7785
Musculoesqueletico     4176
Nervioso               3904
Cardiovascular         2344
Hematopoyetico         1407
Name: count, dtype: int64


---
## 2. Construcción Train – Test

### 2.1 Estrategia de división

Para construir los conjuntos de entrenamiento y prueba a partir de M se adoptan los siguientes principios:

**División por donante (`GroupShuffleSplit` con `SUBJID`):**  
Cada muestra GTEx pertenece a un donante específico. El mismo donante puede contribuir muestras de múltiples tejidos. Una división aleatoria por muestra permite que el mismo donante aparezca en train y test simultáneamente, introduciendo un sesgo: el modelo aprende perfiles individuales de expresión génica en lugar de patrones generalizables. La corrección es dividir por `SUBJID`, garantizando `Tri ∩ Tsi = ∅` a nivel de donante.

**Proporción 80/20 y preservación de probabilidades de ocurrencia:**  
El porcentaje de división (80% train, 20% test) se determina de forma que, al dividir cada partición $M_i$, no se desvíe la probabilidad de ocurrencia de los patrones. Con la división por donante y 946 donantes únicos (756 train / 190 test), la proporción de muestras de cada grupo de tejido se preserva en ambos conjuntos — verificada explícitamente en la tabla de la sección siguiente.

**Verificación formal:**  
- `Tri ∩ Tsi = ∅` — verificado por ausencia de donantes compartidos entre conjuntos.  
- `⋃ Tri ∪ Tsi = M` — verificado por suma de tamaños igual a |M|.  
- Proporciones por partición — verificadas comparando porcentajes de cada clase en train y test.

In [9]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

# Codificar TISSUE_GROUP (5 clases)
le_tissue = LabelEncoder()
M = M.copy()
M['label'] = le_tissue.fit_transform(M['TISSUE_GROUP'])

X = M[gene_cols].values
y = M['label'].values
groups = M['SUBJID'].values

print(f'Clases (TISSUE_GROUP → label):')
for code_val, name in enumerate(le_tissue.classes_):
    cnt = (y == code_val).sum()
    print(f'  {code_val}: {name} ({cnt:,} muestras)')

Clases (TISSUE_GROUP → label):
  0: Cardiovascular (2,344 muestras)
  1: Hematopoyetico (1,407 muestras)
  2: Musculoesqueletico (4,176 muestras)
  3: Nervioso (3,904 muestras)
  4: Visceral_Metabolico (7,785 muestras)


In [10]:
# --- División train/test por donante (80/20) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

train_donors = set(groups[train_idx])
test_donors  = set(groups[test_idx])
overlap      = train_donors & test_donors

print('=== Verificación de la división ===')
print(f'Train : {len(X_train):,} muestras | {len(train_donors):,} donantes')
print(f'Test  : {len(X_test):,} muestras  | {len(test_donors):,} donantes')
print(f'Total : {len(X_train)+len(X_test):,} == |M| = {len(M):,} → '
      f'{"OK" if len(X_train)+len(X_test)==len(M) else "ERROR"}')
print(f'Donantes en ambos conjuntos: {len(overlap)} → '
      f'{"OK (Tri ∩ Tsi = ∅)" if overlap==set() else "ERROR"}')

=== Verificación de la división ===
Train : 15,620 muestras | 756 donantes
Test  : 3,996 muestras  | 190 donantes
Total : 19,616 == |M| = 19,616 → OK
Donantes en ambos conjuntos: 0 → OK (Tri ∩ Tsi = ∅)


In [11]:
# --- Distribución por clase en cada conjunto (con proporciones) ---
print('Distribución TISSUE_GROUP en train vs test:')
print(f'  {"Clase":<25} {"Train N":>8} {"Train %":>9} {"Test N":>8} {"Test %":>9}')
print(f'  {"-"*65}')
for code_val, name in enumerate(le_tissue.classes_):
    n_tr  = (y_train == code_val).sum()
    n_ts  = (y_test  == code_val).sum()
    pct_tr = n_tr / len(y_train) * 100
    pct_ts = n_ts / len(y_test)  * 100
    print(f'  {name:<25} {n_tr:>8,} {pct_tr:>8.1f}% {n_ts:>8,} {pct_ts:>8.1f}%')
print(f'  {"-"*65}')
print(f'  {"Total":<25} {len(y_train):>8,} {"100.0%":>9} {len(y_test):>8,} {"100.0%":>9}')
print()
print('Las proporciones de cada clase se mantienen entre train y test.')
print('La división por donante no introduce sesgo en la probabilidad de ocurrencia de patrones.')

Distribución TISSUE_GROUP en train vs test:
  Clase                      Train N   Train %   Test N    Test %
  -----------------------------------------------------------------
  Cardiovascular               1,871     12.0%      473     11.8%
  Hematopoyetico               1,123      7.2%      284      7.1%
  Musculoesqueletico           3,311     21.2%      865     21.6%
  Nervioso                     3,096     19.8%      808     20.2%
  Visceral_Metabolico          6,219     39.8%    1,566     39.2%
  -----------------------------------------------------------------
  Total                       15,620    100.0%    3,996    100.0%

Las proporciones de cada clase se mantienen entre train y test.
La división por donante no introduce sesgo en la probabilidad de ocurrencia de patrones.


---
## 3. Selección de métricas para medir calidad de resultados

### 3.1 Consideraciones para grandes volúmenes de datos

Con 19,788 muestras y 5 clases desbalanceadas (Visceral_Metabólico ≫ Hematopoyético), las métricas deben ser:

- **Robustas ante desbalance**: accuracy global es engañoso cuando las clases son desiguales (un clasificador que predice siempre la clase mayoritaria puede alcanzar alta accuracy). F1-macro pondera clases por igual.
- **Interpretables en contexto biológico**: precision y recall por clase revelan qué grupos de tejido son más difíciles de discriminar.
- **Escalables**: PySpark `MulticlassClassificationEvaluator` y `BinaryClassificationEvaluator` calculan métricas directamente sobre DataFrames distribuidos sin recolectar todos los datos.

### 3.2 Métricas para el modelo supervisado (Random Forest — clasificación multi-clase)

| Métrica | Fórmula | Justificación |
|---------|---------|---------------|
| **Accuracy** | (VP+VN)/(VP+VN+FP+FN) | Baseline rápido; útil cuando las clases están balanceadas o como referencia |
| **F1-macro** | Promedio no ponderado de F1 por clase | Penaliza igualmente errores en clases chicas (Hematopoyético, 1,407 muestras) y grandes (Visceral, 7,957) |
| **Precision y Recall por clase** | TP/(TP+FP), TP/(TP+FN) | Revelan confusiones biológicas específicas (ej. cardiovascular vs musculoesquelético) |
| **Matriz de confusión** | N×N tabla de conteos | Diagnóstico visual de errores sistemáticos entre grupos de tejido |

### 3.3 Métricas para el modelo no supervisado (K-Means — clustering)

| Métrica | Descripción | Justificación |
|---------|-------------|---------------|
| **Silhouette** (intrínseco) | Cohesión interna vs separación de clusters | No requiere etiquetas; mide la calidad geométrica del clustering. Rango: [-1, 1]; >0.25 = estructura razonable |
| **WCSS / Inercia** (intrínseco) | Suma de distancias cuadradas al centroide | Usado en el método del codo para seleccionar k óptimo |
| **Pureza** (extrínseco) | Fracción de la clase mayoritaria por cluster | Valida biológicamente el clustering comparando con etiquetas de tejido |

### 3.4 Implementación

Las métricas supervisadas se calculan con `MulticlassClassificationEvaluator` de PySpark MLlib (escalable, sobre el DataFrame distribuido) y `classification_report` de scikit-learn (para detalle por clase, aplicado post-hoc sobre predicciones recolectadas). Las métricas de clustering se calculan con `silhouette_score` de scikit-learn aplicado sobre las predicciones recolectadas con `toPandas()` — estrategia validada en Tarea 4 para evitar el bug de `ClusteringEvaluator` en Windows.

In [12]:
# --- Definición de evaluadores PySpark ---
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy')
f1_evaluator  = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='f1')

from sklearn.metrics import silhouette_score as sk_silhouette, classification_report
from sklearn.metrics import confusion_matrix

print('Evaluadores definidos:')
print('  Supervisado  : MulticlassClassificationEvaluator (accuracy, F1-macro)')
print('  No supervisado: sklearn.metrics.silhouette_score (post-hoc)')

Evaluadores definidos:
  Supervisado  : MulticlassClassificationEvaluator (accuracy, F1-macro)
  No supervisado: sklearn.metrics.silhouette_score (post-hoc)


---
## 4. Entrenamiento de Modelos de Aprendizaje

### 4.1 Estrategia general

Se entrenan dos tipos de modelos complementarios:

| Modelo | Algoritmo | Implementación | Objetivo |
|--------|-----------|----------------|----------|
| **Supervisado** | Random Forest (100 árboles) | PySpark MLlib | Clasificar TISSUE_GROUP (5 clases) |
| **No supervisado** | K-Means (k óptimo por método del codo) | PySpark MLlib + sklearn PCA | Descubrir estructura latente en M |

**Preprocesamiento compartido:**  
- `StandardScaler` (sklearn): normaliza a media=0, std=1 — necesario para K-Means (sensible a escala); no afecta RF.
- `PCA(50 componentes)` (sklearn): reduce dimensionalidad de 500 → 50 features, explicando >90% de la varianza. Evita la maldición de la dimensionalidad en K-Means y acelera el entrenamiento de ambos modelos.

**Prevención de sobreajuste:**  
- Split por donante (sección 2): evaluación sobre muestras de individuos no vistos en entrenamiento.
- RF: `featureSubsetStrategy='sqrt'` introduce aleatoriedad en la selección de features por árbol.
- RF: `maxDepth=10` limita la profundidad máxima de cada árbol.
- PCA antes de K-Means: reduce ruido en features de baja varianza.

In [13]:
# --- Preprocesamiento: StandardScaler + PCA (sklearn, sobre driver) ---
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as SklearnPCA

N_PCA = 50

scaler = StandardScaler(with_mean=True, with_std=True)
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

pca = SklearnPCA(n_components=N_PCA, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca  = pca.transform(X_test_sc)

cum_var = pca.explained_variance_ratio_.cumsum()
print(f'Varianza explicada acumulada:')
for k in [5, 10, 20, 50]:
    print(f'  PC 1–{k:>2}: {cum_var[k-1]*100:.1f}%')
print(f'\nX_train_pca: {X_train_pca.shape}')
print(f'X_test_pca : {X_test_pca.shape}')

Varianza explicada acumulada:
  PC 1– 5: 37.7%
  PC 1–10: 53.7%
  PC 1–20: 67.5%
  PC 1–50: 82.8%

X_train_pca: (15620, 50)
X_test_pca : (3996, 50)


In [14]:
# --- Crear Spark DataFrames con features PCA ---
from pyspark.ml.feature import VectorAssembler

pca_cols = [f'pc{i:02d}' for i in range(N_PCA)]

def to_spark_pca(X_pca, y_arr, label_col='label'):
    df = pd.DataFrame(X_pca, columns=pca_cols)
    df[label_col] = y_arr
    sdf = spark.createDataFrame(df)
    assembler = VectorAssembler(inputCols=pca_cols, outputCol='pca_features')
    return assembler.transform(sdf).select('pca_features', label_col).cache()

train_pca_spark = to_spark_pca(X_train_pca, y_train)
test_pca_spark  = to_spark_pca(X_test_pca,  y_test)

# Spark DataFrame con features originales (sin PCA) para RF supervisado
# train_raw_spark se cachea — RF hace múltiples pasadas sobre los datos durante el entrenamiento
def to_spark_raw(X_arr, y_arr):
    df_pd = pd.DataFrame(X_arr, columns=gene_cols)
    df_pd['label'] = y_arr
    return spark.createDataFrame(df_pd)

train_raw_spark = to_spark_raw(X_train, y_train).cache()
test_raw_spark  = to_spark_raw(X_test,  y_test)

print(f'train_pca_spark: {train_pca_spark.count():,} filas')
print(f'test_pca_spark : {test_pca_spark.count():,} filas')
print(f'train_raw_spark: {train_raw_spark.count():,} filas')

train_pca_spark: 15,620 filas


test_pca_spark : 3,996 filas


train_raw_spark: 15,620 filas


### 4.2 Modelo supervisado: Random Forest

**Hiperparámetros seleccionados:**

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| `numTrees` | 100 | Balance entre estabilidad de predicción y tiempo de cómputo |
| `maxDepth` | 10 | Limita sobreajuste; suficiente para capturar interacciones de genes |
| `featureSubsetStrategy` | `sqrt` | Estándar para clasificación en RF; √500 ≈ 22 features/árbol |
| `seed` | 42 | Reproducibilidad |

**Elección de RF sobre otros algoritmos:**  
RF es insensible a la escala de features (no requiere StandardScaler), robusto ante features irrelevantes (descartadas por la selección aleatoria por árbol), y provee importancia de features para interpretación biológica. Alternativas como SVM o regresión logística requieren normalización y son más lentas en datasets grandes. En literatura de RNA-seq, RF es el clasificador de referencia para clasificación de tejido.

In [15]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

assembler_rf = VectorAssembler(inputCols=gene_cols, outputCol='features')
rf = RandomForestClassifier(
    labelCol='label', featuresCol='features',
    numTrees=100, maxDepth=10,
    featureSubsetStrategy='sqrt',
    seed=RANDOM_SEED
)
pipeline_rf = Pipeline(stages=[assembler_rf, rf])

print('Entrenando Random Forest (5 clases, split por donante)...')
model_rf = pipeline_rf.fit(train_raw_spark)
print('Entrenamiento completado.')

Entrenando Random Forest (5 clases, split por donante)...


Entrenamiento completado.


In [16]:
# --- Evaluación del modelo supervisado ---
preds_rf = model_rf.transform(test_raw_spark)

acc_rf = acc_evaluator.evaluate(preds_rf)
f1_rf  = f1_evaluator.evaluate(preds_rf)

preds_rf_pd = preds_rf.select('label', 'prediction').toPandas()

print(f'Accuracy (macro): {acc_rf:.4f}')
print(f'F1-Score (macro): {f1_rf:.4f}')
print()
print('Reporte por grupo de tejido:')
print(classification_report(
    preds_rf_pd['label'].astype(int),
    preds_rf_pd['prediction'].astype(int),
    target_names=le_tissue.classes_
))

Accuracy (macro): 0.9842
F1-Score (macro): 0.9843

Reporte por grupo de tejido:
                     precision    recall  f1-score   support

     Cardiovascular       1.00      0.98      0.99       473
     Hematopoyetico       1.00      1.00      1.00       284
 Musculoesqueletico       0.95      0.99      0.97       865
           Nervioso       1.00      1.00      1.00       808
Visceral_Metabolico       0.99      0.97      0.98      1566

           accuracy                           0.98      3996
          macro avg       0.99      0.99      0.99      3996
       weighted avg       0.98      0.98      0.98      3996



In [17]:
# --- Matriz de confusión ---
cm = confusion_matrix(
    preds_rf_pd['label'].astype(int),
    preds_rf_pd['prediction'].astype(int)
)
cm_df = pd.DataFrame(
    cm,
    index=[f'Real: {c}' for c in le_tissue.classes_],
    columns=[f'Pred: {c}' for c in le_tissue.classes_]
)
print('Matriz de confusión (test set):')
print(cm_df.to_string())

Matriz de confusión (test set):
                           Pred: Cardiovascular  Pred: Hematopoyetico  Pred: Musculoesqueletico  Pred: Nervioso  Pred: Visceral_Metabolico
Real: Cardiovascular                        465                     0                         1               0                          7
Real: Hematopoyetico                          0                   283                         0               0                          1
Real: Musculoesqueletico                      2                     0                       855               1                          7
Real: Nervioso                                0                     0                         1             806                          1
Real: Visceral_Metabolico                     0                     0                        42               0                       1524


In [18]:
# --- Construir lookup Name → Símbolo génico para los top-500 genes ---
# Lectura rápida: solo columnas Name+Description (sin muestras), finaliza al encontrar los 500 genes
desc_map = {}
for chunk in pd.read_csv(FILE_PATH, sep='\t', skiprows=2,
                          chunksize=CHUNK_SIZE, usecols=['Name', 'Description']):
    hits = chunk[chunk['Name'].isin(set(tpm_top.index))]
    desc_map.update(dict(zip(hits['Name'], hits['Description'])))
    if len(desc_map) >= len(tpm_top.index):
        break

# tpm_top.index tiene IDs con puntos (ENSG00000160781.17) → gene_cols tiene guiones bajos
san_to_symbol = {g.replace('.', '_'): desc_map.get(g, g) for g in tpm_top.index}

# --- Importancia de features (top-20 genes) ---
rf_model_obj = model_rf.stages[-1]
importances  = rf_model_obj.featureImportances.toArray()
feat_imp = sorted(zip(gene_cols, importances), key=lambda x: x[1], reverse=True)

print('Top 20 genes más importantes para clasificar TISSUE_GROUP:')
print(f'{"Rank":<6} {"Símbolo":<15} {"Ensembl (sanitizado)":<32} {"Importancia"}')
print('-' * 70)
for rank, (col, imp) in enumerate(feat_imp[:20], 1):
    symbol = san_to_symbol.get(col, col)
    print(f'{rank:<6} {symbol:<15} {col:<32} {imp:.5f}')

Top 20 genes más importantes para clasificar TISSUE_GROUP:
Rank   Símbolo         Ensembl (sanitizado)             Importancia
----------------------------------------------------------------------
1      PAQR6           ENSG00000160781_17               0.03792
2      MTURN           ENSG00000180354_16               0.03100
3      MBP             ENSG00000197971_16               0.02811
4      PLP1            ENSG00000123560_14               0.02065
5      GFAP            ENSG00000131095_14               0.01645
6      S100B           ENSG00000160307_10               0.01626
7      BGN             ENSG00000182492_16               0.01541
8      IGFBP7          ENSG00000163453_11               0.01416
9      ACTC1           ENSG00000159251_8                0.01342
10     FABP3           ENSG00000121769_8                0.01158
11     MT3             ENSG00000087250_9                0.01142
12     COL3A1          ENSG00000168542_16               0.01139
13     PIGR            ENSG0000016

### 4.3 Modelo no supervisado: K-Means con PCA

**Preprocesamiento:** StandardScaler + PCA(50) aplicado en sección 4.1 — el espacio de 50 componentes captura el 82.8% de la varianza genómica sobre las 19,616 muestras.

**Selección de k:** método del codo con índice Silhouette para k=2..7. Con el dataset completo (19,616 muestras, 5 grupos de tejido heterogéneos) se espera que el k óptimo refleje la diversidad intra-grupo: Visceral_Metabólico (7,785 muestras) agrupa hígado, páncreas, pulmón, riñón y otros órganos con perfiles de expresión génica distintos, por lo que el k óptimo puede superar el número de grupos de tejido.

**Hiperparámetros K-Means:**

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| `initMode` | K-Means++ (PySpark default) | Inicialización determinista que reduce iteraciones hasta convergencia |
| `maxIter` | 50 | Suficiente para convergencia en 50 componentes PCA |
| `seed` | 42 | Reproducibilidad |

In [19]:
from pyspark.ml.clustering import KMeans

silhouette_scores = {}
wcss_scores       = {}

print('Método del codo (Silhouette y WCSS para k=2..7):')
print(f'{"k":<6} {"Silhouette":>12} {"WCSS (train)":>15}')
print('-' * 38)

for k in range(2, 8):
    km = KMeans(
        featuresCol='pca_features', predictionCol='prediction',
        k=k, maxIter=50, seed=RANDOM_SEED
    )
    km_model = km.fit(train_pca_spark)
    wcss = km_model.summary.trainingCost

    preds_pd = km_model.transform(test_pca_spark) \
        .select('pca_features', 'prediction').toPandas()
    y_pred = preds_pd['prediction'].values
    X_eval = np.vstack([v.toArray() for v in preds_pd['pca_features']])

    if len(np.unique(y_pred)) < 2:
        sil = -1.0
    else:
        sil = sk_silhouette(X_eval, y_pred, metric='euclidean')

    silhouette_scores[k] = sil
    wcss_scores[k]       = wcss
    print(f'{k:<6} {sil:>12.4f} {wcss:>15,.0f}')

Método del codo (Silhouette y WCSS para k=2..7):
k        Silhouette    WCSS (train)
--------------------------------------


2            0.1646       5,631,143


3            0.1827       5,249,681


4            0.2016       4,876,595


5            0.1771       4,737,735


6            0.2149       4,215,261


7            0.2283       4,002,651


In [20]:
# --- Entrenamiento final con k óptimo ---
k_opt = max(silhouette_scores, key=silhouette_scores.get)
print(f'k óptimo (mayor Silhouette): k={k_opt}  (Silhouette={silhouette_scores[k_opt]:.4f})')

km_final = KMeans(
    featuresCol='pca_features', predictionCol='prediction',
    k=k_opt, maxIter=50, seed=RANDOM_SEED
)
model_km = km_final.fit(train_pca_spark)
print(f'K-Means entrenado con k={k_opt}.')

k óptimo (mayor Silhouette): k=7  (Silhouette=0.2283)


K-Means entrenado con k=7.


In [21]:
# --- Evaluación del clustering: Silhouette + Pureza ---
preds_km_pd = model_km.transform(test_pca_spark) \
    .select('pca_features', 'label', 'prediction').toPandas()

y_km_pred = preds_km_pd['prediction'].values
X_km_eval = np.vstack([v.toArray() for v in preds_km_pd['pca_features']])
y_true    = preds_km_pd['label'].values

sil_final = sk_silhouette(X_km_eval, y_km_pred, metric='euclidean')

# Pureza: fracción de la clase mayoritaria de tejido por cluster
n_correct = 0
print(f'Silhouette (k={k_opt}): {sil_final:.4f}')
print('\nTabla de contingencia cluster vs TISSUE_GROUP:')
ct = pd.crosstab(
    preds_km_pd['prediction'],
    preds_km_pd['label'].map(dict(enumerate(le_tissue.classes_))),
    rownames=['Cluster'],
    colnames=['TISSUE_GROUP']
)
print(ct)
for cluster_id in ct.index:
    n_correct += ct.loc[cluster_id].max()
purity = n_correct / len(preds_km_pd)
print(f'\nPureza del clustering: {purity:.4f} ({n_correct}/{len(preds_km_pd)} muestras correctas)')

Silhouette (k=7): 0.2283

Tabla de contingencia cluster vs TISSUE_GROUP:
TISSUE_GROUP  Cardiovascular  Hematopoyetico  Musculoesqueletico  Nervioso  \
Cluster                                                                      
0                        288              55                 277       135   
1                         31               1                  11       673   
2                          0              60                 132         0   
3                          0               0                 162         0   
4                        154               0                   0         0   
5                          0               0                 283         0   
6                          0             168                   0         0   

TISSUE_GROUP  Visceral_Metabolico  
Cluster                            
0                             819  
1                             543  
2                              55  
3                               0  
4       

In [22]:
# --- Tabla resumen de resultados ---
print('=' * 60)
print('  RESUMEN DE RESULTADOS — ETAPA 3')
print('=' * 60)
print()
print('MODELO SUPERVISADO — Random Forest (TISSUE_GROUP, 5 clases)')
print(f'  Accuracy (macro) : {acc_rf:.4f}')
print(f'  F1-Score (macro) : {f1_rf:.4f}')
print(f'  Split            : por donante (SUBJID), 0 solapamiento')
print()
print('MODELO NO SUPERVISADO — K-Means')
print(f'  k óptimo         : {k_opt}')
print(f'  Silhouette       : {sil_final:.4f}')
print(f'  Pureza           : {purity:.4f}')
print('=' * 60)

  RESUMEN DE RESULTADOS — ETAPA 3

MODELO SUPERVISADO — Random Forest (TISSUE_GROUP, 5 clases)
  Accuracy (macro) : 0.9842
  F1-Score (macro) : 0.9843
  Split            : por donante (SUBJID), 0 solapamiento

MODELO NO SUPERVISADO — K-Means
  k óptimo         : 7
  Silhouette       : 0.2283
  Pureza           : 0.5983


---
## 5. Análisis de resultados

### 5.1 Modelo supervisado — Random Forest

**Resultados obtenidos:**

| Clase | Precision | Recall | F1 | Soporte |
|-------|-----------|--------|----|---------|
| Cardiovascular | 1.00 | 0.98 | 0.99 | 473 |
| Hematopoyetico | 1.00 | 1.00 | 1.00 | 284 |
| Musculoesquelético | 0.95 | 0.99 | 0.97 | 865 |
| Nervioso | 1.00 | 1.00 | 1.00 | 808 |
| Visceral_Metabólico | 0.99 | 0.97 | 0.98 | 1,566 |
| **Macro avg** | **0.99** | **0.99** | **0.99** | **3,996** |

**Accuracy: 98.42% · F1-macro: 98.43%**

**Fortalezas:**

- **Alta discriminación con 5 clases:** el modelo clasifica correctamente los 5 grupos de tejido con 98.4% de accuracy sobre 3,996 muestras de 190 donantes no vistos durante el entrenamiento. Esto valida que el perfil de expresión de 500 genes de alta varianza es suficiente para identificar el origen tisular de una muestra RNA-seq incluso en individuos nuevos.
- **Clase más difícil — Musculoesquelético (precision=0.95):** los errores principales son confusiones entre Musculoesquelético y Visceral_Metabólico (42 muestras mal clasificadas en la matriz de confusión). Biológicamente consistente: tejido adiposo comparte genes metabólicos con hígado y páncreas.
- **Generalización robusta:** la división por `SUBJID` (756 donantes en train, 190 en test, 0 solapamiento) garantiza que las métricas reflejan capacidad de predicción en individuos nuevos — condición clave para aplicaciones médicas.

**Áreas de oportunidad:**

- **Desbalance de clases:** Visceral_Metabólico (7,785 muestras) cuadruplica a Hematopoyético (1,407). Aunque F1-macro pondera clases igualmente, class-weighted training o oversampling (SMOTE) podría mejorar la detección de la clase minoritaria.
- **Granularidad SMTSD:** `TISSUE_GROUP` agrupa ~50 sub-tejidos en 5 categorías. Para mayor resolución biológica (ej. ventrículo vs aurícula, músculo esquelético vs fibroblastos), clasificar por SMTSD sería más informativo.
- **Ajuste de hiperparámetros:** `numTrees=100, maxDepth=10` son valores heurísticos. Un grid search con `GroupKFold` por donante podría identificar configuraciones óptimas.

### 5.2 Modelo no supervisado — K-Means

**Resultados obtenidos:**

| k | Silhouette | Observación |
|---|-----------|-------------|
| 2 | 0.1646 | Poca estructura — el espacio génico no es bipartito |
| 3 | 0.1827 | Mejora moderada |
| 4 | 0.2016 | Estructura emergente |
| 5 | 0.1771 | Descenso — k=5 no captura la sub-estructura del dataset completo |
| 6 | 0.2149 | Recuperación |
| **7** | **0.2283** | **k óptimo** |

**K-Means k=7: Silhouette=0.2283, Pureza=59.83%**

**Fortalezas:**

- **k óptimo = 7:** con la muestra completa (19,616 muestras), la estructura interna excede los 5 grupos de tejido. k=7 sugiere sub-grupos dentro de Visceral_Metabólico (7,785 muestras: hígado, páncreas, pulmón, riñón, órganos digestivos con perfiles de expresión génica distintos).
- **Descubrimiento sin etiquetas:** pureza del 59.83% sobre 3,996 muestras indica estructura real (vs ~14% para asignación aleatoria a 7 clusters), confirmando que el perfil génico contiene señal suficiente para agrupar tejidos sin supervisión.

**Áreas de oportunidad:**

- **Silhouette moderado (0.2283):** el espacio de 50 PCs captura el 82.8% de la varianza. Con el dataset completo la dispersión intra-grupo es mayor, dificultando la separación geométrica. Aumentar a 100 PCs o aplicar UMAP antes del clustering podría mejorar la cohesión.
- **Pureza limitada por Visceral_Metabólico:** este grupo heterogéneo domina varios clusters por su tamaño, reduciendo la pureza global. Un clustering jerárquico o espectral capturaría mejor la diversidad intra-grupo.
- **GMM como alternativa:** K-Means asume clusters esféricos. GMM modelaría mejor la geometría elíptica en el espacio PCA, aunque con mayor costo computacional sobre 19,616 muestras.

### 5.3 Síntesis e implicaciones para medicina espacial

Los resultados establecen una **línea base robusta del transcriptoma humano** en condiciones normales:

- El modelo supervisado (RF, 98.4% accuracy) demuestra que 500 genes de alta varianza son suficientes para identificar el origen tisular de cualquier muestra RNA-seq con alta certeza, incluso en individuos nunca vistos durante el entrenamiento.
- El modelo no supervisado (K-Means, k=7) revela que la complejidad biológica del transcriptoma humano excede la agrupación en 5 grupos macroscópicos: hay sub-estructura fina dentro de grupos grandes como Visceral_Metabólico.

Para el objetivo de medicina espacial, estos modelos permiten:
1. **Detección de anomalías en astronautas:** muestras con perfil de expresión que no encaja en ningún cluster normal indicarían cambios fisiológicos inducidos por microgravedad o radiación.
2. **Localización del tejido afectado:** el clasificador supervisado puede señalar qué grupo tisular muestra mayor desviación respecto a la línea base terrestre de GTEx.
3. **Extensibilidad sin re-entrenamiento:** nuevas muestras (ej. datos de astronautas post-vuelo) pueden compararse directamente contra los centroides aprendidos sobre la población GTEx.

---
## Referencias

1. Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324
2. GTEx Consortium. (2020). The GTEx Consortium atlas of genetic regulatory effects across human tissues. *Science*. https://doi.org/10.1126/science.aaz1776
3. GTEx Portal. (2025). GTEx Analysis V10 Downloads. Broad Institute. https://gtexportal.org/home/downloads/adult-gtex
4. Law, C. W., et al. (2016). voom: Precision weights unlock linear model analysis tools for RNA-seq read counts. *Genome Biology*, 17, 29.
5. Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825–2830.
6. Zaharia, M., et al. (2016). Apache Spark: A Unified Engine for Big Data Processing. *Communications of the ACM*, 59(11), 56–65.